In [45]:
import sys
sys.path.insert(0, '../')
sys.path.insert(1, '../../')
from build_config import ALL_SEASONS, COMPETITIONS, CURRENT_SEASON, INPUT_CSV_PATHS, TARGET_RANGES
from const import REPO_PATH, RAW_DATA_PATH, PROCESSED_DATA_PATH, MODELS_PATH
from fbref_const import URLs, TARGET_COLUMNS

from src.data.match_data_processing import process_match_target_var, process_match_other_var
from src.feature.feature_encoders import TeamEncoder, TeamLagFeatureGenerator, PreviousSeasonTeamAverager, TeamRestDaysCalculator, TeamLagTargetFeature

import pandas as pd
import os
import numpy as np
import mlflow
import joblib
import tempfile
import os

from build_utils import *

In [46]:
key_columns=['date', 'home', 'away']

In [47]:
team_encoder_path = f"{MODELS_PATH}/data_processors/{COMPETITIONS[0]}/team_encoder.pkl"
encoder = TeamEncoder.load(team_encoder_path)

In [73]:
encoder.encoder.transform([['Aston Villa', 'Tottenham']])

/Users/tianqihuang/anaconda3/envs/betbot/lib/python3.12/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


array([[0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
        0., 0., 0., 0., 0., 1., 0., 0., 0., 0.]])

In [76]:
encoder.encoder.categories_[0].shape

(21,)

In [48]:
seasons=sorted(ALL_SEASONS)

In [49]:
test_dict=dict((k, pd.read_csv(f'{REPO_PATH}/{INPUT_CSV_PATHS[k]}')) for k in COMPETITIONS)

In [50]:
all_features_dict={}
for competition in COMPETITIONS:
    if competition not in test_dict:
        continue
    test_df = test_dict[competition]
    team_encoder_path = f"{MODELS_PATH}/data_processors/{competition}/team_encoder.pkl"
    season_dfs_dict = {season: pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_data_df.csv") for season in seasons}
    target_dfs = [pd.read_csv(f"{PROCESSED_DATA_PATH}/{competition}/{season}/all_target_df.csv") for season in seasons]
    # TeamEncoder features
    encoder = TeamEncoder.load(team_encoder_path)
    team_encoder_features = encoder.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagFeatureGenerator features
    lag_feature_generator = TeamLagFeatureGenerator(lookback=5)
    team_lag_features = lag_feature_generator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # PreviousSeasonTeamAverager features
    prev_season_averager = PreviousSeasonTeamAverager(decay_factor=1, date_col='date', home_col='home', away_col='away')
    prev_season_features = prev_season_averager.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamRestDaysCalculator features
    rest_days_calculator = TeamRestDaysCalculator()
    rest_days_features = rest_days_calculator.transform_spot(season_dfs_dict, test_df).reset_index(drop=True)

    # TeamLagTargetFeature features
    lag_target_generator = TeamLagTargetFeature(lookback=5)
    lag_target_features = lag_target_generator.transform_spot(season_dfs_dict, target_dfs, test_df).reset_index(drop=True)

    # Merge all features on home, away, date
    from functools import reduce
    feature_dfs = [team_encoder_features, team_lag_features, prev_season_features, rest_days_features, lag_target_features]
    all_features = reduce(lambda left, right: pd.merge(left, right, on=['home', 'away', 'date'], how='outer'), feature_dfs)
    all_features_dict[competition] = all_features.copy().fillna(-1)

In [66]:
train_features=pd.read_csv(f"{REPO_PATH}/data/features/premier_league/all_combined_features_2017-24.csv")
train_features['date'] = pd.to_datetime(train_features['date'])
train_features = train_features.merge(all_features_dict['premier_league'][['home', 'away', 'date']], on=['home', 'away', 'date'], how='right').fillna(-1)

for competition, df in all_features_dict.items():
    all_features_dict[competition] = df[train_features.columns]

In [68]:
# Compare train_features and all_features_dict['premier_league'] (excluding home, away, date)
tf_values = train_features.drop(columns=['home', 'away', 'date']).values
af_values = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).values

# Find where they differ
diff_mask = 1-np.isclose(tf_values, af_values)
diff_indices = np.argwhere(diff_mask)
print(f"Number of differing values: {diff_indices.shape[0]}")
print("First 10 differences:")
train_feature_cols = train_features.drop(columns=['home', 'away', 'date']).columns.tolist()
all_feature_cols = all_features_dict['premier_league'].drop(columns=['home', 'away', 'date']).columns.tolist()
for idx, (row, col) in enumerate(diff_indices[:10]):
    home = train_features.iloc[row]['home']
    away = train_features.iloc[row]['away']
    date = train_features.iloc[row]['date']
    tf_val = tf_values[row, col]
    af_val = af_values[row, col]
    col_name = train_feature_cols[col]
    col_name_af = all_feature_cols[col]

    print(f"Row {row}, Column '{col_name}, {col_name_af}': home={home}, away={away}, date={date}, train_features={tf_val}, all_features_dict={af_val}")

Number of differing values: 4
First 10 differences:
Row 1, Column 'encoded_home_new_team_home, encoded_home_new_team_home': home=Aston Villa, away=Tottenham, date=2020-02-16 00:00:00, train_features=0.0, all_features_dict=1.0
Row 6, Column 'encoded_home_new_team_home, encoded_home_new_team_home': home=Norwich City, away=Liverpool, date=2020-02-15 00:00:00, train_features=0.0, all_features_dict=1.0
Row 7, Column 'encoded_home_new_team_home, encoded_home_new_team_home': home=Sheffield Utd, away=Bournemouth, date=2020-02-09 00:00:00, train_features=0.0, all_features_dict=1.0
Row 9, Column 'encoded_home_new_team_home, encoded_home_new_team_home': home=Wolves, away=Leicester City, date=2020-02-14 00:00:00, train_features=0.0, all_features_dict=1.0


In [55]:
train_features=train_features.merge(all_features_dict['premier_league'][['home', 'away', 'date']], on=['home', 'away', 'date'], how='right').fillna(-1)

In [7]:
model_dict={
    'premier_league': {
        'away_goals':{
            'run_id': '96ae06339d4e4d608dc8c43ee41909b1',
            'artifact_path': 'model',
        },
        'home_goals':{
            'run_id': 'ba14f52e3c4f420da6b0a15e6031045c',
            'artifact_path': 'model',
        },
    }
}

In [8]:
def load_joblib_model_from_mlflow(run_id, artifact_path, tracking_uri=None):
    """
    Fetch and load a joblib-dumped model from MLflow given experiment_id, run_id, and artifact_path.
    Optionally specify the MLflow tracking URI.
    Returns the loaded model.
    """
    if tracking_uri is not None:
        mlflow.set_tracking_uri(tracking_uri)
    client = mlflow.tracking.MlflowClient()
    # Download artifact to a temporary directory
    with tempfile.TemporaryDirectory() as tmp_dir:
        local_path = client.download_artifacts(run_id, artifact_path, tmp_dir)
        # Find the first .joblib file in the artifact directory
        for root, _, files in os.walk(local_path):
            for file in files:
                if file.endswith('.joblib'):
                    model_path = os.path.join(root, file)
                    return joblib.load(model_path)
        raise FileNotFoundError("No .joblib model file found in the artifact path.")


In [9]:
predictions={}

In [14]:
for competition in COMPETITIONS:
    predictions[competition] = {}
    for target_name in TARGET_RANGES:
        model = load_joblib_model_from_mlflow(model_dict[competition][target_name]['run_id'],
                                                model_dict[competition][target_name]['artifact_path'],
                                                f'{REPO_PATH}/mlflow')
        prediction = model.predict_proba(all_features_dict[competition].drop(columns=key_columns))
        prediction = pd.DataFrame(prediction, columns=list(range(TARGET_RANGES[target_name][0]))+['other'])
        predictions[competition][target_name]=prediction

In [15]:
# Add home, away, date columns from test_dict to each predictions DataFrame
for competition in predictions:
    test_rows = test_dict[competition][['home', 'away', 'date']].reset_index(drop=True)
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        # Prepend home, away, date columns
        df = pd.concat([test_rows, df.reset_index(drop=True)], axis=1)
        predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [19]:
model.class_mapping

{0: (-inf, 0, '≤ 0'),
 1: (0, 1, '1-1'),
 2: (1, 2, '2-2'),
 3: (2, 3, '3-3'),
 4: (3, 4, '4-4'),
 5: (4, 5, '5-5'),
 6: (5, 6, '6-6'),
 7: (6, inf, '> 6')}

In [16]:
predictions['premier_league']['home_goals']

,home,away,date,0,1,2,3,4,5,other
0,Everton,Crystal Palace,2020-02-08,4.378595e-27,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
1,Brighton,Watford,2020-02-08,9.793509e-01,0.012438,0.003377,0.003502,0.000731,0.000194,0.000407
2,Sheffield Utd,Bournemouth,2020-02-09,1.834220e-25,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
3,Wolves,Leicester City,2020-02-14,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Southampton,Burnley,2020-02-15,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,Norwich City,Liverpool,2020-02-15,1.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
6,Aston Villa,Tottenham,2020-02-16,9.793509e-01,0.012438,0.003377,0.003502,0.000731,0.000194,0.000407
7,Arsenal,Newcastle Utd,2020-02-16,9.793509e-01,0.012438,0.003377,0.003502,0.000731,0.000194,0.000407
8,Chelsea,Manchester Utd,2020-02-17,0.000000e+00,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000
9,Manchester City,West Ham,2020-02-19,0.000000e+00,0.000000,1.000000,0.000000,0.000000,0.000000,0.000000


In [17]:
predictions['premier_league']['away_goals']

,home,away,date,0,1,2,3,4,5,6,other
0,Everton,Crystal Palace,2020-02-08,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,Brighton,Watford,2020-02-08,0.001336,0.910994,0.066142,0.019183,0.000381,0.000405,0.000372,0.001187
2,Sheffield Utd,Bournemouth,2020-02-09,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
3,Wolves,Leicester City,2020-02-14,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
4,Southampton,Burnley,2020-02-15,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,Norwich City,Liverpool,2020-02-15,0.977875,0.000000,0.000000,0.000000,0.022125,0.000000,0.000000,0.000000
6,Aston Villa,Tottenham,2020-02-16,0.001336,0.910994,0.066142,0.019183,0.000381,0.000405,0.000372,0.001187
7,Arsenal,Newcastle Utd,2020-02-16,0.001336,0.910994,0.066142,0.019183,0.000381,0.000405,0.000372,0.001187
8,Chelsea,Manchester Utd,2020-02-17,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
9,Manchester City,West Ham,2020-02-19,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [ ]:
# Rename columns and update values in predictions DataFrames for each competition and target
for competition in predictions:
    for target_name in predictions[competition]:
        df = predictions[competition][target_name]
        cols = df.columns.tolist()
        # Only rename and update non-metadata columns (assume first three are home, away, date)
        meta_cols = ['home', 'away', 'date']
        feature_cols = cols[3:]
        # Rename columns: first becomes lte_{original}, others become gt_{previous}
        new_cols = meta_cols.copy()
        if feature_cols:
            new_cols.append(f"lte_{feature_cols[0]}")
            for i in range(1, len(feature_cols)):
                new_cols.append(f"gt_{feature_cols[i-1]}")
        # Update values: first feature column stays, others become sum of itself and all to the right
        arr = df[feature_cols].values.copy() if feature_cols else None
        if arr is not None and arr.shape[1] > 0:
            for i in range(1, arr.shape[1]):
                arr[:, i] = arr[:, i:].sum(axis=1)
            df = pd.concat([df[meta_cols].reset_index(drop=True), pd.DataFrame(arr, columns=new_cols[3:])], axis=1)
            df.columns = new_cols
            predictions[competition][target_name] = df

# Example: predictions['premier_league']['home_goals'].head()

In [ ]:
predictions['premier_league']['home_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5
0,Everton,Crystal Palace,2020-02-08,0.174223,0.825777,0.679403,0.529377,0.382203,0.215885,0.137942
1,Brighton,Watford,2020-02-08,0.180657,0.819343,0.667427,0.471577,0.316344,0.162016,0.081453
2,Sheffield Utd,Bournemouth,2020-02-09,0.175834,0.824166,0.683844,0.543897,0.398783,0.250893,0.130970
3,Wolves,Leicester City,2020-02-14,0.172842,0.827158,0.678835,0.529994,0.389799,0.218760,0.139779
4,Southampton,Burnley,2020-02-15,0.166940,0.833060,0.684672,0.535765,0.392639,0.221351,0.142335
5,Norwich City,Liverpool,2020-02-15,0.184205,0.815795,0.663036,0.508955,0.372829,0.221729,0.142578
6,Aston Villa,Tottenham,2020-02-16,0.160736,0.839264,0.700076,0.532579,0.381470,0.245600,0.128207
7,Arsenal,Newcastle Utd,2020-02-16,0.167836,0.832164,0.693421,0.502440,0.367394,0.208854,0.134299
8,Chelsea,Manchester Utd,2020-02-17,0.172262,0.827738,0.670555,0.507620,0.349578,0.168323,0.084624
9,Manchester City,West Ham,2020-02-19,0.168323,0.831677,0.675434,0.517404,0.343090,0.165112,0.083010


In [ ]:
predictions['premier_league']['away_goals']

,home,away,date,lte_0,gt_0,gt_1,gt_2,gt_3,gt_4,gt_5,gt_6
0,Everton,Crystal Palace,2020-02-08,0.160206,0.839794,0.695910,0.531938,0.416352,0.306474,0.195714,0.108582
1,Brighton,Watford,2020-02-08,0.154920,0.845080,0.703231,0.551090,0.432150,0.302140,0.192946,0.107047
2,Sheffield Utd,Bournemouth,2020-02-09,0.152552,0.847448,0.710438,0.559629,0.453118,0.327543,0.206148,0.103395
3,Wolves,Leicester City,2020-02-14,0.164249,0.835751,0.693161,0.523961,0.413239,0.304350,0.193954,0.107606
4,Southampton,Burnley,2020-02-15,0.162472,0.837528,0.695630,0.539442,0.429466,0.302873,0.193012,0.107083
5,Norwich City,Liverpool,2020-02-15,0.158338,0.841662,0.699456,0.546933,0.433868,0.303531,0.193432,0.107316
6,Aston Villa,Tottenham,2020-02-16,0.160961,0.839039,0.715069,0.542637,0.429565,0.320309,0.196637,0.109094
7,Arsenal,Newcastle Utd,2020-02-16,0.153914,0.846086,0.707853,0.550320,0.437202,0.310507,0.188028,0.104318
8,Chelsea,Manchester Utd,2020-02-17,0.154839,0.845161,0.707919,0.538435,0.431417,0.308978,0.186678,0.103569
9,Manchester City,West Ham,2020-02-19,0.161660,0.838340,0.694020,0.529552,0.417228,0.307402,0.196306,0.108911
